# Resume Intelligence — Automated Resume Parser
Extracts structured data (personal info, links, skills, education, experience, projects) from a PDF resume and predicts a likely job role, then exports everything to JSON.

## Section 1 - Install Libraries

In [ ]:
!pip install -q PyMuPDF
!python -m spacy download en_core_web_lg -q

## Section 2 - Imports

In [ ]:
import fitz          # PyMuPDF
import spacy
import re
import json
from spacy.pipeline import EntityRuler
from google.colab import files

nlp = spacy.load("en_core_web_lg")
print("spaCy model loaded successfully ✅")

## Section 3 - Upload Resume

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
print(f"Uploaded file: {filename}")

## Section 4 - PDF Text Extraction

In [ ]:
def extract_text_from_pdf(path):
    """Extract raw text from every page of the PDF resume."""
    pdf = fitz.open(path)
    text = ""
    for page in pdf:
        text += page.get_text()
    pdf.close()
    return text

raw_text = extract_text_from_pdf(filename)
print(raw_text)

## Section 5 - Text Cleaning

In [ ]:
def clean_text(text):
    """Normalize whitespace and line breaks so downstream regex/NER is reliable."""
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)      # collapse repeated spaces/tabs
    text = re.sub(r"\n{2,}", "\n", text)      # collapse repeated blank lines
    text = text.strip()
    return text

resume_text = clean_text(raw_text)
print(resume_text)

## Section 6 - Personal Information

In [ ]:
doc = nlp(resume_text)

def extract_name(doc, fallback_text):
    """Use spaCy NER to find a PERSON entity; fall back to the first line of the resume."""
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            # Some PDFs merge the name with the following line (e.g. "Arvind Kumar\nLinkedIn")
            candidate = ent.text.split("\n")[0].strip()
            if len(candidate.split()) <= 4:  # avoid grabbing whole sentences
                return candidate
    return fallback_text.strip().split("\n")[0]

name = extract_name(doc, resume_text)

email_pattern = r"[\w\.-]+@[\w\.-]+\.\w+"
emails = re.findall(email_pattern, resume_text)

phone_pattern = r"(\+?\d[\d\s\-\(\)]{8,}\d)"
phones = re.findall(phone_pattern, resume_text)

personal_info = {
    "name": name,
    "email": emails[0] if emails else None,
    "phone": phones[0] if phones else None
}

personal_info

## Section 7 - Social Links

In [ ]:
def extract_hyperlinks(path):
    """Pull actual clickable hyperlinks embedded in the PDF (LinkedIn, GitHub, etc.)."""
    pdf = fitz.open(path)
    links = []
    for page in pdf:
        for link in page.get_links():
            uri = link.get("uri")
            if uri:
                links.append(uri)
    pdf.close()
    return list(dict.fromkeys(links))  # de-duplicate, preserve order

def extract_plaintext_urls(text):
    """Fallback: catch any bare URLs written as plain text in the resume."""
    url_pattern = r"(https?://[^\s|]+|www\.[^\s|]+)"
    return re.findall(url_pattern, text)

all_links = extract_hyperlinks(filename) + extract_plaintext_urls(resume_text)
all_links = list(dict.fromkeys(all_links))

def classify_links(links):
    social = {"linkedin": None, "github": None, "leetcode": None, "portfolio": None, "other": []}
    for link in links:
        l = link.lower()
        if "linkedin.com" in l:
            social["linkedin"] = link
        elif "github.com" in l:
            social["github"] = link
        elif "leetcode.com" in l:
            social["leetcode"] = link
        elif any(k in l for k in ["portfolio", "behance", "vercel", "netlify"]):
            social["portfolio"] = link
        else:
            social["other"].append(link)
    return social

social_links = classify_links(all_links)
social_links

## Section 8 - Skills Extraction

In [ ]:
# Register a custom EntityRuler so spaCy recognizes technical skills as entities
if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
else:
    ruler = nlp.get_pipe("entity_ruler")

skill_terms = [
    "Python", "Java", "JavaScript", "TypeScript", "SQL", "HTML", "CSS",
    "React", "React Native", "Next.js", "Redux", "Recharts",
    "Node.js", "Express", "Express.js", "FastAPI", "Django", "Flask",
    "MongoDB", "PostgreSQL", "MySQL", "Redis", "Prisma", "Firebase",
    "Docker", "Kubernetes", "AWS", "AWS EC2", "Git", "GitHub",
    "Spring Boot", "TensorFlow", "PyTorch", "Scikit-learn", "Pandas",
    "NumPy", "Matplotlib", "Machine Learning", "Deep Learning", "NLP",
    "Expo", "Google Maps API", "Multer", "EJS", "OOP", "DBMS",
    "Operating Systems", "Computer Networks", "Data Structures & Algorithms"
]

skill_patterns = [{"label": "SKILL", "pattern": term} for term in skill_terms]
ruler.add_patterns(skill_patterns)

# Re-run the pipeline now that the ruler has the skill patterns loaded
doc = nlp(resume_text)

skills = sorted({ent.text for ent in doc.ents if ent.label_ == "SKILL"})
skills

## Section 9 - Education Extraction

In [ ]:
education_keywords = [
    "B.Tech", "Bachelor", "Master", "M.Tech", "B.E", "BSc", "MSc", "PhD", "Diploma"
]

degrees_found = [kw for kw in education_keywords if kw.lower() in resume_text.lower()]

# Isolate the Education block so we can pull institution / duration details too
edu_match = re.search(
    r"Education\s*(.*?)(?=Technical Skills|Skills|Experience|Projects|$)",
    resume_text, re.DOTALL | re.IGNORECASE
)
education_block = edu_match.group(1).strip() if edu_match else ""

date_range_pattern = r"([A-Za-z]{3,9}\s\d{4})\s*[–-]\s*([A-Za-z]{3,9}\s\d{4}|Present)"
edu_dates = re.findall(date_range_pattern, education_block)

education = {
    "degrees": degrees_found,
    "details": education_block,
    "duration": f"{edu_dates[0][0]} - {edu_dates[0][1]}" if edu_dates else None
}

education

## Section 10 - Experience Extraction

In [ ]:
experience_keywords = [
    "Intern", "Internship", "Software Engineer", "Developer",
    "Full Stack", "Backend", "Frontend"
]

experience = [kw for kw in experience_keywords if kw.lower() in resume_text.lower()]
experience

## Section 11 - Project Extraction

In [ ]:
# Isolate the Projects block
proj_match = re.search(
    r"Projects\s*(.*?)(?=Achievements|Certifications|Publications|$)",
    resume_text, re.DOTALL | re.IGNORECASE
)
projects_block = proj_match.group(1).strip() if proj_match else ""

lines = [line.strip() for line in projects_block.split("\n") if line.strip()]

projects = []
current = None

for line in lines:
    # A project header line looks like: "Title | Tech, Stack | GitHub"
    if "|" in line and not line.startswith("•"):
        if current:
            projects.append(current)
        parts = [p.strip() for p in line.split("|")]
        title = parts[0]
        tech_stack = [t.strip() for t in parts[1].split(",")] if len(parts) > 1 else []
        link_hint = parts[2] if len(parts) > 2 else None
        current = {
            "title": title,
            "tech_stack": tech_stack,
            "link_hint": link_hint,
            "description": []
        }
    elif line.startswith("•") and current:
        current["description"].append(line.lstrip("•").strip())

if current:
    projects.append(current)

projects

## Section 12 - Role Prediction

In [ ]:
# Simple rule-based role predictor: score each candidate role by keyword overlap
role_keywords = {
    "Frontend Developer": ["React", "JavaScript", "HTML", "CSS", "Next.js", "Redux", "TypeScript"],
    "Backend Developer": ["Node.js", "Express", "FastAPI", "MongoDB", "PostgreSQL", "MySQL",
                           "Redis", "Java", "Spring Boot"],
    "Full Stack Developer": ["React", "Node.js", "Express", "MongoDB", "PostgreSQL", "JavaScript",
                              "FastAPI"],
    "Mobile App Developer": ["React Native", "Expo", "Firebase", "Google Maps API"],
    "Machine Learning Engineer": ["Python", "Machine Learning", "Deep Learning", "TensorFlow",
                                   "PyTorch", "Scikit-learn", "NLP", "Pandas", "NumPy"],
    "DevOps Engineer": ["Docker", "Kubernetes", "AWS", "AWS EC2", "Git", "GitHub"]
}

def predict_role(skills, full_text):
    combined_text = (" ".join(skills) + " " + full_text).lower()
    scores = {}
    for role, keywords in role_keywords.items():
        scores[role] = sum(1 for kw in keywords if kw.lower() in combined_text)
    predicted_role = max(scores, key=scores.get)
    return predicted_role, scores

predicted_role, role_scores = predict_role(skills, resume_text)

print("Predicted Role:", predicted_role)
role_scores

## Section 13 - JSON Output

In [ ]:
resume_json = {
    "personal_info": personal_info,
    "social_links": social_links,
    "skills": skills,
    "education": education,
    "experience": experience,
    "projects": projects,
    "predicted_role": {
        "role": predicted_role,
        "scores": role_scores
    }
}

print(json.dumps(resume_json, indent=4))

## Section 14 - Save JSON

In [ ]:
output_filename = "resume_data.json"

with open(output_filename, "w") as f:
    json.dump(resume_json, f, indent=4)

print(f"Resume data saved to {output_filename} ✅")

files.download(output_filename)